<a href="https://colab.research.google.com/github/RajeevKambhatla/longevity-drug-repurposing/blob/main/07_llm_gene_lists.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q anthropic pydantic tenacity gseapy biopython
from google.colab import drive, userdata
drive.mount('/content/drive')

import json, os, time, pandas as pd
BASE = '/content/drive/MyDrive/longevity_project'
os.makedirs(f'{BASE}/data/processed/llm_genes', exist_ok=True)

hallmarks = json.load(open(f'{BASE}/data/processed/hallmarks.json'))
genes = pd.read_csv(f'{BASE}/data/raw/GSE92742_Broad_LINCS_gene_info.txt.gz', sep='\t')
bing = genes[genes['pr_is_bing'] == 1]
id_to_symbol = dict(zip(bing['pr_gene_id'].astype(str), bing['pr_gene_symbol']))
symbols = set(id_to_symbol.values())
print(list(hallmarks))
vcap = pd.read_csv(f'{BASE}/data/processed/vcap_sig_info.csv')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 31.3 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['senescence', 'nutrient_sensing', 'mitochondria', 'autophagy', 'inflammation', 'proteostasis']


In [ ]:
from pydantic import BaseModel
from typing import Literal

class GeneEntry(BaseModel):
    gene: str
    confidence: Literal['core', 'supporting', 'speculative']
    role: str

class HallmarkGenes(BaseModel):
    hallmark: str
    genes: list[GeneEntry]

In [ ]:
import anthropic
from tenacity import retry, stop_after_attempt, wait_exponential
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

DESCRIPTIONS = {
    'senescence': 'cellular senescence and the senescence-associated secretory phenotype (genes higher in senescent cells)',
    'nutrient_sensing': 'mTORC1 signaling (genes induced when mTORC1 is active)',
    'mitochondria': 'mitochondrial oxidative phosphorylation and biogenesis (genes higher when mitochondrial function is high)',
    'autophagy': 'macroautophagy (genes induced when autophagy is activated, for example by TFEB or FOXO)',
    'inflammation': 'chronic inflammatory signaling (genes induced by NF-kB and related inflammatory pathways)',
    'proteostasis': 'the unfolded protein response and proteotoxic stress (genes induced under protein-folding stress)',
}

SYSTEM = """You are an expert in the biology of aging. For the process described, list human genes whose mRNA
levels rise when that process is more active in human cells, so the list can be used to read the process from gene expression data.
Use official HGNC gene symbols. Give 30 to 60 genes, favoring well-established ones. Tag each gene's confidence as
core, supporting, or speculative, with a one-sentence role. Respond with only a JSON object with keys hallmark and genes
(a list of objects with keys gene, confidence, role)."""

@retry(stop=stop_after_attempt(3), wait=wait_exponential(min=2, max=20))
def propose_genes(name):
    msg = client.messages.create(model='claude-sonnet-5', max_tokens=8000, system=SYSTEM,
        messages=[{'role': 'user', 'content': f'Hallmark: {name}\nProcess: {DESCRIPTIONS[name]}'}])
    raw = next(b.text for b in msg.content if b.type == 'text')
    raw = raw.strip().removeprefix('```json').removesuffix('```').strip()
    return HallmarkGenes.model_validate_json(raw)

In [ ]:
for name in hallmarks:
    path = f'{BASE}/data/processed/llm_genes/{name}.json'
    if os.path.exists(path):
        continue
    result = propose_genes(name)
    open(path, 'w').write(result.model_dump_json(indent=2))
    print(name, len(result.genes), 'genes')

senescence 51 genes
nutrient_sensing 45 genes
mitochondria 54 genes
autophagy 51 genes
inflammation 49 genes
proteostasis 51 genes


In [ ]:
candidates = []
for name, hm in hallmarks.items():
    llm = json.load(open(f'{BASE}/data/processed/llm_genes/{name}.json'))['genes']
    msig = set(hm['genes'])
    in_data = [g for g in llm if g['gene'] in symbols]
    overlap = [g for g in in_data if g['gene'] in msig]
    new_core = [g for g in in_data if g['gene'] not in msig and g['confidence'] == 'core']
    print(f"{name:17} LLM genes in data: {len(in_data):3} | already in MSigDB: {len(overlap):3} | new core genes: {len(new_core)}")
    candidates += [{'hallmark': name, **g} for g in new_core]

candidates = pd.DataFrame(candidates)

senescence        LLM genes in data:  49 | already in MSigDB:  35 | new core genes: 7
nutrient_sensing  LLM genes in data:  43 | already in MSigDB:  20 | new core genes: 7
mitochondria      LLM genes in data:  41 | already in MSigDB:  26 | new core genes: 4
autophagy         LLM genes in data:  43 | already in MSigDB:  31 | new core genes: 2
inflammation      LLM genes in data:  47 | already in MSigDB:  20 | new core genes: 4
proteostasis      LLM genes in data:  46 | already in MSigDB:  18 | new core genes: 5


In [ ]:
from Bio import Entrez
Entrez.email = 'your.email@example.com'
TERMS = {'senescence': 'senescence', 'nutrient_sensing': 'mTORC1', 'mitochondria': 'oxidative phosphorylation',
         'autophagy': 'autophagy', 'inflammation': 'inflammation', 'proteostasis': 'unfolded protein response'}

def paper_count(gene, hallmark):
    q = f'{gene}[Title/Abstract] AND "{TERMS[hallmark]}"[Title/Abstract]'
    return int(Entrez.read(Entrez.esearch(db='pubmed', term=q, retmax=0))['Count'])

counts = []
for _, c in candidates.iterrows():
    counts.append(paper_count(c['gene'], c['hallmark']))
    time.sleep(0.4)
candidates['papers'] = counts
pd.set_option('display.max_colwidth', 80)
candidates.sort_values(['hallmark', 'papers'], ascending=[True, False])[['hallmark', 'gene', 'papers', 'role']]

,hallmark,gene,papers,role
18,autophagy,TFEB,1808,Master transcription factor that activates autophagy and lysosomal biogenesi...
19,autophagy,FOXO3,378,Transcription factor inducing autophagy gene expression under stress/caloric...
20,inflammation,TNF,84585,Pro-inflammatory cytokine induced by and amplifying NF-kB signaling.
22,inflammation,PTGS2,1264,"Encodes COX-2, strongly induced by NF-kB during inflammation."
21,inflammation,TNFAIP3,406,"Encodes A20, NF-kB-induced negative regulator reflecting active NF-kB signal..."
23,inflammation,BIRC3,63,NF-kB-induced anti-apoptotic gene commonly used as marker of pathway activity.
15,mitochondria,TFAM,273,Mitochondrial transcription factor essential for mtDNA replication and trans...
16,mitochondria,NRF1,137,Nuclear respiratory factor driving transcription of OXPHOS and biogenesis ge...
14,mitochondria,PPARGC1A,76,Master transcriptional coactivator driving mitochondrial biogenesis and OXPH...
17,mitochondria,POLG,68,"Catalytic subunit of mtDNA polymerase, central to mitochondrial genome repli..."


In [ ]:
keep = {
    'senescence': ['CDKN2A', 'CDKN1A', 'CDKN2B', 'GLB1', 'CEBPB', 'NFKB1'],
    'nutrient_sensing': ['EIF4EBP1', 'SLC3A2', 'SREBF1'],
    'mitochondria': ['TFAM', 'NRF1', 'PPARGC1A', 'POLG'],
    'autophagy': ['TFEB'],
    'inflammation': ['TNF', 'PTGS2', 'TNFAIP3', 'BIRC3'],
    'proteostasis': ['DDIT3', 'PPP1R15A', 'TRIB3', 'HSPA1A', 'DNAJB1'],
}

for name, hm in hallmarks.items():
    hm['genes_refined'] = sorted(set(hm['genes']) | set(keep[name]))
    hm['added_by_llm'] = keep[name]
    print(f"{name:17} {len(hm['genes'])} -> {len(hm['genes_refined'])} genes")
json.dump(hallmarks, open(f'{BASE}/data/processed/hallmarks.json', 'w'), indent=2)

senescence        125 -> 131 genes
nutrient_sensing  200 -> 203 genes
mitochondria      200 -> 204 genes
autophagy         335 -> 336 genes
inflammation      200 -> 204 genes
proteostasis      113 -> 118 genes


In [19]:
pilot = pd.read_csv(f'{BASE}/data/processed/senescence_pilot_drugs.csv')
named = vcap[~vcap['pert_iname'].str.startswith('BRD-')].copy()
named['name'] = named['pert_iname'].str.lower()

stress = ['tunicamycin', 'thapsigargin', 'bortezomib', 'geldanamycin', 'tanespimycin', 'mg-132']
stress_here = [d for d in stress if d in set(named['name'])]
extra = named[named['name'].isin(stress_here)].copy()
extra['is_24h'] = extra['pert_time'] == 24
extra = extra.sort_values(['is_24h', 'pert_dose'], ascending=False).groupby('name').head(1)

test = pd.concat([pilot, extra]).drop_duplicates('name')
print('Stress drugs found:', stress_here, '| Total drugs:', len(test))

Stress drugs found: ['tunicamycin', 'thapsigargin', 'bortezomib', 'geldanamycin', 'tanespimycin', 'mg-132'] | Total drugs: 106


In [20]:
sigs = pd.read_parquet(f'{BASE}/data/processed/vcap_signatures.parquet', columns=list(id_to_symbol))
sigs = sigs.loc[test['sig_id']]
sigs.columns = [id_to_symbol[c] for c in sigs.columns]
print(sigs.shape)

(106, 10174)


In [21]:
import gseapy as gp
from tqdm import tqdm

gene_sets = {name: hm['genes_refined'] for name, hm in hallmarks.items()}
rows = []
for sig_id, row in tqdm(sigs.iterrows(), total=len(sigs)):
    res = gp.prerank(rnk=row.sort_values(ascending=False), gene_sets=gene_sets,
                     permutation_num=100, min_size=5, max_size=1000,
                     seed=0, outdir=None, verbose=False)
    nes = res.res2d.set_index('Term')['NES'].astype(float)
    rows.append({'sig_id': sig_id, **nes.to_dict()})

scores = pd.DataFrame(rows).set_index('sig_id')
for name, hm in hallmarks.items():
    if hm['good'] == 'down':
        scores[name] = -scores[name]
scores = scores.join(test.set_index('sig_id')['name']).set_index('name')
scores.round(3).to_csv(f'{BASE}/results/six_hallmark_pilot_refined.csv')
print(scores.shape)

  0%|          | 0/106 [00:00<?, ?it/s]2026-10-06 20:56:39,298 [WARNING] Duplicated values found in preranked stats: 0.02% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
  1%|          | 1/106 [00:00<00:21,  4.91it/s]2026-10-06 20:56:39,485 [WARNING] Duplicated values found in preranked stats: 0.02% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
  2%|▏         | 2/106 [00:00<00:20,  5.13it/s]2026-10-06 20:56:39,673 [WARNING] Duplicated values found in preranked stats: 0.04% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
  3%|▎         | 3/106 [00:00<00:18,  5.60it/s]2026-10-06 20:56:39,834 [WARNING] Duplicated values found in preranked stats: 5.52% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
  4%|▍         | 4/106 [00:00<00:17,  5.76it/s]2026-10-06 20:56:40,001 [WARNING] Duplicated values found in preranked st

(106, 6)


In [ ]:
changes = []
for d, hm in controls:
    moved = old_r.loc[d, hm] - new_r.loc[d, hm]   # positive = moved toward the top
    if hm == 'proteostasis':
        moved = -moved                            # stress drugs should move toward the bottom
    changes.append(moved)
    print(f'{d:14} {hm:17} {old_r.loc[d, hm]:3} -> {new_r.loc[d, hm]:3}  ({"better" if moved > 0 else "worse" if moved < 0 else "same"})')

avg = sum(changes) / len(changes)
verdict = 'improved' if avg > 3 else 'worsened' if avg < -3 else 'did not change'
print(f'\nAverage movement in the right direction: {avg:.1f} places -> {verdict}')

NameError: name 'controls' is not defined

In [22]:
old = pd.read_csv(f'{BASE}/results/six_hallmark_pilot.csv', index_col='name')
new = pd.read_csv(f'{BASE}/results/six_hallmark_pilot_refined.csv', index_col='name')
old_r, new_r = old.rank(ascending=False).astype(int), new.rank(ascending=False).astype(int)

controls = [('sirolimus', 'nutrient_sensing')] + [(d, 'proteostasis') for d in stress_here] + \
           [(d, 'senescence') for d in ['navitoclax', 'quercetin', 'metformin', 'resveratrol'] if d in new.index]
for d, hm in controls:
    print(f'{d:14} {hm:17} rank before: {old_r.loc[d, hm]:3}  after: {new_r.loc[d, hm]:3}')

sirolimus      nutrient_sensing  rank before:   3  after:   3
tunicamycin    proteostasis      rank before: 106  after: 105
thapsigargin   proteostasis      rank before:  34  after:  30
bortezomib     proteostasis      rank before:  84  after:  98
geldanamycin   proteostasis      rank before:  99  after: 106
tanespimycin   proteostasis      rank before:  23  after:  33
mg-132         proteostasis      rank before:  61  after:  77
navitoclax     senescence        rank before:  31  after:  33
quercetin      senescence        rank before:  51  after:  61
metformin      senescence        rank before:  41  after:  59
resveratrol    senescence        rank before:  43  after:  53


In [23]:
import json
hallmarks = json.load(open(f'{BASE}/data/processed/hallmarks.json'))
use_refined = {'proteostasis'}
for name, hm in hallmarks.items():
    hm['genes_final'] = hm['genes_refined'] if name in use_refined else hm['genes']
    print(f"{name:17} using {'refined' if name in use_refined else 'original'} list ({len(hm['genes_final'])} genes)")
json.dump(hallmarks, open(f'{BASE}/data/processed/hallmarks.json', 'w'), indent=2)

senescence        using original list (125 genes)
nutrient_sensing  using original list (200 genes)
mitochondria      using original list (200 genes)
autophagy         using original list (335 genes)
inflammation      using original list (200 genes)
proteostasis      using refined list (118 genes)
